#  Pandas ve NumPy ile  Veri Analizi

Bu ders, algoritmik trading botları geliştirmek isteyenler için pandas ve NumPy kütüphanelerini kapsamlı bir şekilde ele alır. Bu kaynak sadece kodları değil, arka planda çalışan **finansal mantığı ve algoritmik tuzakları** da öğretmeyi amaçlar.

Ders sonunda gerçek fiyat verisi üzerinde teknik analiz yapabilecek, `shift()`, `cumprod()`, `astype()` gibi kritik fonksiyonların felsefesini kavrayacak, makinelere sinyal ürettirebilecek ve kendi ticaret stratejilerinizi (Backtest) geçmişe dönük olarak test edebileceksiniz.

### Ders İçeriği

1. Kütüphaneleri İçe Aktarma
2. NumPy Temelleri ve Finansal Matematik
3. pandas Temelleri — Tabloların Kontrol Merkezi
4. Gerçek Veri Çekme — yfinance API
5. Veri İnceleme, Temizleme ve "Küçülen Tablo Tuzağı"
6. Veri Seçme ve Şartlı Filtreleme
7. Teknik Göstergelerin (İndikatörlerin) İnşası
8. Sinyal Üretme (Makinelere Karar Aldırmak)
9. Strateji Testi (Backtest) ve Zaman Makinesi Mantığı
10. Görselleştirme

---
## 1. Kütüphaneleri İçe Aktarma

Her Python projesinde kullanacağımız kütüphaneleri en başta import ederiz. `pd` ve `np` kısaltmaları tüm dünyada kullanılan standarttır.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("Laboratuvar Hazır!")

---
## 2. NumPy Temelleri

NumPy (Numerical Python), bilimsel hesaplamanın temelidir. pandas'ın altında çalışır. Trading botlarında fiyat hesaplamaları, gösterge formülleri hep NumPy ile yapılır.

### 2.1 Array (Dizi) Oluşturma

NumPy'nin temel veri yapısı `ndarray`'dir. Python listelerinden çok daha hızlı çalışır çünkü verileri bellekte bitişik olarak saklar.

In [ ]:
# Listeden array oluşturma
fiyatlar = np.array([150.0, 152.3, 148.5, 155.0, 160.2])
print("Array:", fiyatlar)
print("Tip:", type(fiyatlar))
print("Boyut:", fiyatlar.shape)
print("Eleman sayısı:", len(fiyatlar))

In [ ]:
# Özel array'ler
print(np.zeros(5))        # beş sıfır
print(np.ones(5))         # beş bir
print(np.arange(0, 10, 2))  # 0'dan 10'a kadar 2'şer artarak
print(np.linspace(0, 1, 5)) # 0 ile 1 arasında 5 eşit aralıklı sayı

### 2.2 Matematiksel İşlemler

NumPy'nin gücü, tüm elemanlar üzerinde tek satırda işlem yapabilmesidir. Normal Python listesiyle bunu yapmak için for döngüsü gerekirdi.

In [ ]:
fiyatlar = np.array([150.0, 152.3, 148.5, 155.0, 160.2])

# Tüm elemanlara işlem uygulanır — for döngüsü gerekmez
print("Ortalama:        ", round(np.mean(fiyatlar), 2))
print("Medyan:          ", round(np.median(fiyatlar), 2))
print("Standart sapma (Volatilite):  ", round(np.std(fiyatlar), 2))
print("Minimum:         ", np.min(fiyatlar))
print("Maximum:         ", np.max(fiyatlar))
print("Toplam:          ", np.sum(fiyatlar))

**Algoritmik Trader Notu:** Piyasada fiyatın kendisinden çok, **fiyatın değişimi** önemlidir. NumPy'daki `np.diff()` ardışık elemanlar arasındaki matematiksel farkı (Bugün - Dün) bulur.

In [ ]:
# Günlük getiri hesaplama — trading'de çok kullanılır
# (bugünkü fiyat - dünkü fiyat) / dünkü fiyat
fiyatlar = np.array([100.0, 102.0, 101.0, 105.0, 103.0])

# np.diff() ardışık elemanlar arasındaki farkı hesaplar
gunluk_degisim = np.diff(fiyatlar)
gunluk_getiri = np.diff(fiyatlar) / fiyatlar[:-1] * 100

print("Fiyatlar:          ", fiyatlar)
print("Günlük değişim:    ", gunluk_degisim)
print("Günlük getiri (%): ", np.round(gunluk_getiri, 2))

### 2.3 Boolean İndeksleme

Koşula göre eleman seçmek için kullanılır. Trading'de "fiyat şu değerin üzerinde mi?" gibi sinyaller üretmek için kullanırsınız.

In [ ]:
fiyatlar = np.array([100.0, 102.0, 101.0, 105.0, 103.0, 98.0, 107.0])

# 103 üzerindeki fiyatlar
maske = fiyatlar > 103
print("Boolean maske (True/False):", maske)
print("103 üzeri fiyatlar:", fiyatlar[maske])
print("Kaç gün 103 üzerinde:", np.sum(maske))

### 🔬 ALIŞTIRMA - LAB 1
Aşağıdaki `fiyatlar_lab` dizisinden:
1. 101'in altındaki fiyatları filtreleyin ve ekrana basın.
2. Fiyatların aritmetik ortalamasını bulun.
3. Sadece ortalamanın üzerinde olan fiyatları filtreleyip ekrana basın.

In [ ]:
fiyatlar_lab = np.array([100.0, 102.0, 101.0, 105.0, 103.0, 98.0, 107.0, 99.5, 100.2])

# Görev 1 Kodlarını Buraya Yazın:



---
## 3. pandas Temelleri

pandas, NumPy'nin üzerine kurulu, etiketli veri yapıları sağlayan kütüphanedir. NumPy sayılarla ilgilenirken, Pandas onlara tarih, etiket ve isim verir.

- **Series**: Tek boyutlu, etiketli dizi (bir sütun)
- **DataFrame**: İki boyutlu, sütun yönelimli tablo (Excel gibi)

### 3.1 Series

In [ ]:
# Series oluşturma
fiyat_serisi = pd.Series(
    [150.0, 152.3, 148.5, 155.0, 160.2],
    index=["Pzt", "Sal", "Car", "Per", "Cum"]
)

print(fiyat_serisi)
print()
print("Salı fiyatı:", fiyat_serisi["Sal"])
print("Ortalama:   ", round(fiyat_serisi.mean(), 2))

### 3.2 DataFrame Oluşturma

DataFrame'i dict'ten (sözlük) oluşturabiliriz. Key sütun adı, value liste veya array'dir.

In [ ]:
data = {
    "ticker":  ["AAPL", "NVDA", "TSLA", "GOOGL"],
    "sektor":  ["Teknoloji", "Yarı İletken", "EV", "Teknoloji"],
    "adet":    [10, 5, 8, 3],
    "alis":    [150.0, 100.0, 200.0, 140.0],
    "guncel":  [210.0, 205.0, 180.0, 175.0]
}

df = pd.DataFrame(data)
print(df)

In [ ]:
# Temel bilgiler
print("Boyut:", df.shape)        # (satır, sütun)
print("Sütunlar:", df.columns.tolist())
print("Tipler:")
print(df.dtypes)

In [ ]:
# Yeni sütun ekleme ve DataFrame Matematiği
df["kar_dolar"]  = (df["guncel"] - df["alis"]) * df["adet"]
df["kar_pct"]    = ((df["guncel"] - df["alis"]) / df["alis"]) * 100
df["piyasa_deger"] = df["guncel"] * df["adet"]

print(df.round(2))

**Kodlama Pratiği: `==` yerine `.eq()` Kullanımı**
Veri bilimciler işlemleri İngilizce bir cümle gibi soldan sağa okumak isterler. `df["sektor"] == "Teknoloji"` yazmak yerine `df["sektor"].eq("Teknoloji")` yazmak (zincirleme metodolojisi) çok daha profesyonel ve hatasız bir kodlama standardıdır.

In [ ]:
# Teknoloji sektöründeki hisselerin sayısını bulalım:
teknoloji_sayisi = df["sektor"].eq("Teknoloji").sum()
print(f"Teknoloji sektöründeki hisse sayısı: {teknoloji_sayisi}")

---
## 4. Gerçek Veri Çekme — yfinance

`yf.download()` fonksiyonu Yahoo Finance'den geçmiş fiyat verisi çeker ve direkt DataFrame olarak döner.

Dönen DataFrame'deki sütunlar:
- **Open**: Gün açılış fiyatı
- **High**: Günün en yüksek fiyatı
- **Low**: Günün en düşük fiyatı
- **Close**: Gün kapanış fiyatı — teknik analizde en çok kullanılan (Referans Fiyat)
- **Volume**: İşlem hacmi (El değiştiren hisse adedi)

**ÖNEMLİ:** Borsa İstanbul hisseleri için ticker'ın sonuna `.IS` ekleyin. Örn: `THYAO.IS`

In [ ]:
# Tek hisse senedi verisi
df = yf.download("NVDA", start="2024-01-01", end="2024-12-31")

# Bazen yfinance MultiIndex (çift katmanlı başlık) döndürebilir, bunu düzlüyoruz:
if isinstance(df.columns, pd.MultiIndex):
    df.columns = df.columns.get_level_values(0)

print("Boyut:", df.shape)
print("\nİlk 5 satır:")
print(df.head())

In [ ]:
# Birden fazla hisse aynı anda çekilebilir
tickers = ["AAPL", "NVDA", "TSLA"]
df_coklu = yf.download(tickers, start="2024-01-01", end="2024-12-31")

# Sadece kapanış fiyatları
kapanislar = df_coklu["Close"]
print(kapanislar.head())

---
## 5. Veri İnceleme ve Temizleme (Hayati Önem Taşır)

Gerçek dünyada veriler her zaman temiz gelmez. Eksik değerler (Tatiller vb.) veya gösterge hesaplamalarından doğan `NaN` değerleri (İlk N gün) algoritmaları çökertir.

### 5.1 Veriyi Tanıma

In [ ]:
df = yf.download("NVDA", start="2023-01-01", end="2024-12-31")
if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

print("--- GENEL BİLGİ ---")
print(f"Satır sayısı: {len(df)}")
print(f"Tarih aralığı: {df.index[0].date()} → {df.index[-1].date()}")
print()
print("--- İSTATİSTİKSEL ÖZET ---")
print(df.describe().round(2))

In [ ]:
# info() çok önemli — tipleri ve eksik değerleri gösterir
df.info()

### 5.2 Eksik Değerler (NaN) ve "Küçülen Tablo Tuzağı"

NaN (Not a Number) eksik veriyi temsil eder. Örneğin 50 günlük hareketli ortalama hesaplarken ilk 49 gün mecburen `NaN` kalır.

**⚠️ ÖLÜMCÜL TUZAK:** Strateji test ederken veriyi `df.dropna(inplace=True)` ile temizlerseniz, ana tablonuzdan 50 satırı **kalıcı olarak** silmiş olursunuz. İkinci bir strateji test etmek istediğinizde tablo baştan 50 satır daha küçülür! Getiri hesaplamalarınız (% kârlarınız) tamamen bozulur.

**Doğru Yaklaşım:** Ana tabloya (df) asla dokunmamalı, temizleme işlemi için tablonun bağımsız bir **kopyasını (`.copy()`)** oluşturmalıyız.

In [ ]:
df["MA50"] = df["Close"].rolling(50).mean()

print("Eksik değer sayıları:")
print(df.isnull().sum())

# DOĞRU TEMİZLEME YÖNTEMİ (KOPYA OLUŞTURMAK)
test_df = df.dropna().copy()
print(f"\nOrijinal tablonun satır sayısı (Korundu): {len(df)}")
print(f"Test tablosunun (Kopya) satır sayısı: {len(test_df)}")

---
## 6. Veri Seçme ve Filtreleme

DataFrame'den istediğimiz veriyi seçmek için iki yöntem vardır:
- **`loc[]`**: Etiket bazlı seçim (tarih, sütun adı ile)
- **`iloc[]`**: Pozisyon bazlı seçim (sıra numarası ile)

### 6.1 Satır Seçme — loc ve iloc

In [ ]:
# iloc — pozisyon ile seçim (0'dan başlar)
print("İlk satır (iloc[0]):")
print(df.iloc[0][["Close", "Volume"]])

print("\nEn güncel / Son satır (iloc[-1]):")
print(df.iloc[-1][["Close", "Volume"]])

### 6.2 Koşullu Filtreleme (Piyasa Taraması)

Trading sinyalleri üretirken hissenin sadece belirli kurallara uyduğu günleri ayırmak için kullanırız.

In [ ]:
# Kapanış fiyatı 100 doların üzerinde olan günler
yuksek_gunler = df[df["Close"] > 100]
print(f"100$ üzerinde kapanan gün sayısı: {len(yuksek_gunler)}")
print(yuksek_gunler[["Close", "Volume"]].head())

# Birden fazla koşul — & (ve), | (veya)
# Hacim ortalamanın 2 katından fazla VE fiyat 100$ üzeri olan günler
ortalama_hacim = df["Volume"].mean()
yüksek_hacim_ve_fiyat = df[
    (df["Volume"] > ortalama_hacim * 2) &
    (df["Close"] > 100)
]
print(f"\nİki Koşulu sağlayan gün sayısı: {len(yüksek_hacim_ve_fiyat)}")

### 🔬 ALIŞTIRMA - LAB 2 (Fiyat Filtreleme)
Kapanış fiyatının (Close), açılış fiyatından (Open) **yüksek** olduğu günleri filtreleyin (Yükseliş / Yeşil Mum günleri). 
Toplam kaç tane yükseliş günü olduğunu ekrana basın.

In [ ]:
# Görev 2 Kodlarını Buraya Yazın:



---
## 7. Teknik Göstergeler (İndikatörler)

Teknik göstergeler, fiyat verilerinden türetilen matematiksel hesaplardır. Trading botlarının kararlarını bunlara göre verir.

### 7.1 Basit (MA) ve Üstel (EMA) Hareketli Ortalama
- **`rolling(window=20)`**: Klasik. Son 20 günü eşit ağırlıkla değerlendirip ortalama alır. Eğilimleri (Trendleri) gösterir.
- **`ewm(span=20, adjust=False)`**: Üstel (Akıllı) ortalama. Dünkü fiyata, 20 gün önceki fiyattan daha çok değer (ağırlık) verir. Sert fiyat patlamalarına çok daha **hızlı tepki verir**.

In [ ]:
df["MA20"]  = df["Close"].rolling(window=20).mean()
df["MA50"]  = df["Close"].rolling(window=50).mean()
df["EMA12"] = df["Close"].ewm(span=12, adjust=False).mean()
df["EMA26"] = df["Close"].ewm(span=26, adjust=False).mean()

print(df[["Close", "MA20", "MA50", "EMA12", "EMA26"]].tail(5).round(2))

### 7.2 MACD (Trend ve Momentum İkisi Bir Arada)
MACD, iki EMA (Hızlı 12 ve Yavaş 26) arasındaki farkı ölçer. Dünyadaki en meşhur borsa indikatörüdür.
- **MACD Çizgisi**: EMA12 - EMA26
- **Sinyal Çizgisi**: Çıkan MACD'nin kendi 9 günlük EMA'sı (Daha yumuşak bir tetikleyici)
- **Histogram**: MACD - Sinyal (Çubuklar)

In [ ]:
df["MACD"]       = df["EMA12"] - df["EMA26"]
df["Sinyal"]     = df["MACD"].ewm(span=9, adjust=False).mean()
df["Histogram"]  = df["MACD"] - df["Sinyal"]

print(df[["MACD", "Sinyal", "Histogram"]].tail(5).round(4))

### 7.3 RSI (Göreceli Güç Endeksi) & `clip()` Fonksiyonunun Sırrı
RSI, hissenin yükseliş gücünü düşüş gücüyle kıyaslayıp 0 ile 100 arasında puanlar. 
- Burada `clip(lower=0)` fonksiyonunu kullanırız. Amacı: Fiyat listesindeki eksileri (düşüşleri) tıraşlayıp sadece "saf yükselişleri" bir sepete koymaktır.

In [ ]:
def hesapla_rsi(seri, periyot=14):
    delta = seri.diff()                         # günlük değişim (Örn: +5, -3, +2)
    kazan = delta.clip(lower=0)                 # Sadece artıları bırak (Örn: 5, 0, 2)
    kaybet = -delta.clip(upper=0)               # Sadece eksileri bırak ve pozitife çevir (Örn: 0, 3, 0)
    
    ort_kazan = kazan.rolling(periyot).mean()
    ort_kaybet = kaybet.rolling(periyot).mean()
    
    rs = ort_kazan / ort_kaybet
    rsi = 100 - (100 / (1 + rs))
    return rsi

df["RSI"] = hesapla_rsi(df["Close"])
print(df[["Close", "RSI"]].tail(5).round(2))

### 7.4 Bollinger Bantları ve İstatistiksel Oynaklık (`std()`)
Fiyatın etrafındaki oynaklık (volatilite) kanalını çizer. Burada omurga (Orta Bant) MA20'dir. Kanalın ne kadar esneyeceğini ise standart sapma `std()` belirler. İstatistiksel olarak fiyatların %95'i bu iki bandın (2 standart sapma) arasında kalmak zorundadır.

In [ ]:
df["BB_Orta"]  = df["Close"].rolling(20).mean()
df["BB_Std"]   = df["Close"].rolling(20).std()
df["BB_Ust"]   = df["BB_Orta"] + (2 * df["BB_Std"])
df["BB_Alt"]   = df["BB_Orta"] - (2 * df["BB_Std"])

print(df[["Close", "BB_Ust", "BB_Orta", "BB_Alt"]].tail(5).round(2))

### 🔬 ALIŞTIRMA- LAB 3 (Kendi Özel İndikatörünü Yaz)
Klasik göstergeleri kullandınız, sıra kendi kurallarınızı koymakta. 
Öyle bir özel gösterge (`Super_Guc`) oluşturun ki; Kapanış fiyatı, MA20'nin üzerinde olduğu ve aynı zamanda RSI'ın 50'den büyük olduğu günlerde değeri **1**, diğer günlerde değeri **0** olsun.

In [ ]:
# Görev 3 Kodlarını Buraya Yazın:



---
## 8. Sinyal Üretme: Makinelere Karar Aldırmak

Trading botunuz ekrana bakıp "MA20, MA50'yi kesmiş" diyemez. Bunu sayılara (`1` ve `0`) dökmeniz gerekir. 

- **`astype(int)` Ne İşe Yarar?:** Mantıksal sorguların (`df["MA20"] > df["MA50"]`) sonucu Doğru/Yanlış (`True/False`) çıkar. Bunu `astype(int)` ile sararsak `True = 1` (Hisse al), `False = 0` (Hisse sat/Nakite geç) şeklinde pozisyon kuralına dönüşür.
- **`diff()` ile Tetiği Çekmek:** Elimizde 0, 0, 0, 1, 1, 1 listesi varsa, bizim için değerli olan o ilk "1"in (Değişim gününün) yandığı tarihtir. `diff()` sütunları birbirinden çıkararak değişimi yakalar. 1'e geçiş **+1 (AL)**, 0'a geçiş **-1 (SAT)** sinyalidir.

In [ ]:
df_sinyal = df.dropna().copy()

# 1. Pozisyonu Belirle (1: Hissede kal, 0: Nakitte kal)
df_sinyal["MA_Pozisyon"] = (df_sinyal["MA20"] > df_sinyal["MA50"]).astype(int)

# 2. Geçişleri Yakala (1 veya -1)
df_sinyal["MA_Tetik"] = df_sinyal["MA_Pozisyon"].diff()

# Daha önce öğrendiğimiz .eq() metoduyla sinyalleri sayalım
al_sayisi = df_sinyal["MA_Tetik"].eq(1).sum()
sat_sayisi = df_sinyal["MA_Tetik"].eq(-1).sum()

print(f"Golden Cross Stratejisi -> AL Sinyali: {al_sayisi} | SAT Sinyali: {sat_sayisi}")

### 🔬 ALIŞTIRMA- LAB 4 
Yukarıdaki mantığı kullanarak bir RSI botu kurgulayın. Kural:
- RSI değeri 30'un altında ise Pozisyon `1` olsun.
- Değilse Pozisyon `0` olsun.
- Tetiği (`diff()`) çekin ve kaç kere AL, kaç kere SAT sinyali ürettiğini saydırıp yazdırın.

In [ ]:
# Görev 4 Kodlarını Buraya Yazın:



---
## 9. Strateji Testi (Backtest) ve "Zaman Makinesi" Mantığı

Bir stratejinin gerçekten para kazandırıp kazandırmadığını test etmenin en hayati bölümü burasıdır. Bu bölümde iki devasa kavramı öğreneceğiz:

### 1. Geleceği Görme Hatası ve `shift(1)` Çözümü
Pazartesi akşamı piyasa kapandı. Hesap yaptınız ve MACD AL sinyali verdi (`Pozisyon = 1`). Eğer bunu doğrudan pazartesinin getirisiyle çarparsanız, yaşanmış bitmiş geçmişin kârını cebinize koyarsınız (İmkansızdır!).
Pazartesi akşamı üretilen sinyal, ancak **SALI GÜNÜNÜN** getirisini hak edebilir. İşte `.shift(1)` komutu, sistemin sinyalini bir alt satıra (ertesi güne) kaydırarak "Geleceği görme" hilesini engeller ve sizi gerçek piyasa kurallarına bağlar.

### 2. Kümülatif (Bileşik) Getiri ve `cumprod()`
Borsada getiriler toplanmaz (`sum()`), çarpılır (`cumprod()`). Çünkü dünün kârı bugünün ana parasına eklenir (Bileşik Getiri). Formül: `(1 + Getiri).cumprod()`

In [ ]:
def backtest_calistir(ham_veri, kisa_periyot=20, uzun_periyot=50):
    # KURAL 1: Güvenli test ortamı için verinin bağımsız bir kopyasını çıkar ve temizle
    test_df = ham_veri.copy()
    test_df["MA_Kisa"] = test_df["Close"].rolling(kisa_periyot).mean()
    test_df["MA_Uzun"] = test_df["Close"].rolling(uzun_periyot).mean()
    test_df.dropna(inplace=True)
    
    # KURAL 2: Pozisyon kurgula
    test_df["Pozisyon"] = (test_df["MA_Kisa"] > test_df["MA_Uzun"]).astype(int)
    
    # KURAL 3: Günlük yüzdesel (% bazında) fiyat değişimini hesapla
    test_df["Gunluk_Getiri"] = test_df["Close"].pct_change()
    
    # KURAL 4: Zaman Makinesi (shift) - Dünkü kararı bugünün getirisine uygula!
    test_df["Strateji_Getiri"] = test_df["Gunluk_Getiri"] * test_df["Pozisyon"].shift(1)
    
    # KURAL 5: Bileşik Getiri Hesaplaması (Anaparayı 1 birim kabul ederek)
    al_tut   = (1 + test_df["Gunluk_Getiri"]).cumprod().iloc[-1]
    strateji = (1 + test_df["Strateji_Getiri"]).cumprod().iloc[-1]
    
    print(f"Al & Tut (Piyasaya Dokunmama) Getirisi : %{round((al_tut - 1) * 100, 1)}")
    print(f"MA Kesişim Robotunun Getirisi          : %{round((strateji - 1) * 100, 1)}")
    
    return test_df

# Fonksiyonu tetikleyelim
sonuc_tablosu = backtest_calistir(df_coklu["NVDA"].to_frame(name="Close"))

### 🔬 ALIŞTIRMA - LAB 5 
Yukarıdaki Backtest fonksiyonunu kullanarak `AAPL` (Apple) veya `TSLA` (Tesla) hissesini test edin. 


In [ ]:
# Görev 5 Kodlarını Buraya Yazın:



---
## 10. Görselleştirme 



In [ ]:
# Backtest sistemimizden gelen saf tablo üzerinde bileşik getirileri hesaplayıp çizelim
sonuc_tablosu["Al_Tut_Egrisi"] = (1 + sonuc_tablosu["Gunluk_Getiri"]).cumprod()
sonuc_tablosu["Strateji_Egrisi"] = (1 + sonuc_tablosu["Strateji_Getiri"]).cumprod()

plt.figure(figsize=(14, 6))
plt.plot(sonuc_tablosu.index, sonuc_tablosu["Al_Tut_Egrisi"],   label="Sadece Al & Tut",  linewidth=2, color='gray', alpha=0.7)
plt.plot(sonuc_tablosu.index, sonuc_tablosu["Strateji_Egrisi"], label="Algoritmik Strateji", linewidth=2, color='blue')

plt.title("NVDA — Robot vs. Yatırımcı Getiri Karşılaştırması")
plt.ylabel("Ana Para Katsayısı (1.0 = Başlangıç Sermayesi)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 🔬 ALIŞTIRMA - LAB 6 
Sıfırdan kendi stratejinizi ve backtestinizi inşa etme zamanı. 
1. Yeni bir hücre oluşturun. `TSLA` verisini çekin veya hazır olanlardan kullanın.
2. Bollinger Bantlarını (MA20 ve Std*2) hesaplayın.
3. Fiyat Alt Bandın altına indiğinde (`< BB_Alt`) Pozisyon `1` (Al), Üst Bandın üzerine çıktığında (`> BB_Ust`) Pozisyon `0` (Sat) olsun. **İpucu:** Bunu bir önceki konudaki `np.where()` ile yapabilirsiniz.
4. `shift(1)` ile strateji getirisini hesaplayın.
5. `cumprod()` ile bileşik getiriyi bulup sonucu ekrana yazdırın.



In [ ]:
# GÖREV 6 (Final Lab) Kodlarını Buraya Yazın:

